In [1]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm
from sklearn.metrics import accuracy_score, roc_auc_score

# ==============================
# Device Setup
# ==============================
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ==============================
# Transforms
# ==============================
transform = A.Compose([
    A.Resize(112, 112),
    A.Normalize(),
    ToTensorV2()
])

# ==============================
# Siamese Network (Shared backbone)
# ==============================
class SiameseNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=5),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=5),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        with torch.no_grad():
            dummy = self.backbone(torch.zeros(1, 3, 112, 112))
            self.flattened = dummy.view(1, -1).shape[1]
        self.fc = nn.Sequential(
            nn.Linear(self.flattened, 512),
            nn.ReLU(),
            nn.Linear(512, 128)
        )

    def forward_once(self, x):
        x = self.backbone(x)
        x = x.view(x.size(0), -1)
        return F.normalize(self.fc(x), p=2, dim=1)  # L2 normalize

    def forward(self, img1, img2):
        return self.forward_once(img1), self.forward_once(img2)

# ==============================
# PGD Attack for Pairs
# ==============================
def pgd_attack(model, img1, img2, label, criterion, eps=0.03, alpha=0.01, iters=10):
    img1_adv = img1.clone().detach().to(device)
    img1_orig = img1.clone().detach().to(device)
    img1_adv.requires_grad = True
    for _ in range(iters):
        out1, out2 = model(img1_adv, img2)
        loss = criterion(out1, out2, label)
        model.zero_grad()
        if img1_adv.grad is not None:
            img1_adv.grad.zero_()
        loss.backward()
        img1_adv = img1_adv + alpha * img1_adv.grad.sign()
        eta = torch.clamp(img1_adv - img1_orig, min=-eps, max=eps)
        img1_adv = torch.clamp(img1_orig + eta, 0, 1).detach()
        img1_adv.requires_grad = True
    return img1_adv

# ==============================
# Contrastive Loss
# ==============================
class ContrastiveLoss(nn.Module):
    def __init__(self, margin=1.0):
        super().__init__()
        self.margin = margin

    def forward(self, output1, output2, label):
        dists = F.pairwise_distance(output1, output2, p=2)
        loss = label * dists.pow(2) + (1 - label) * F.relu(self.margin - dists).pow(2)
        return loss.mean()



/Users/markogolovko/Projects/Adversarial-Attack-Challenge/.venv/lib/python3.12/site-packages/albumentations/__init__.py:28: UserWarning: A new version of Albumentations is available: '2.0.7' (you have '2.0.6'). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


Using device: mps


In [2]:
from datasets.siamese_face_dataset import create_dataloaders_with_partition

# ==============================
# Dataset, Model, Loss, Optimizer
# ==============================

base_path = './AdvLFW/'  # Update as needed
partition_file = os.path.join(base_path, 'list_eval_partition_no_overlap.txt')
image_dir = os.path.join(base_path, 'images')

train_loader, val_loader, test_loader, dataset = create_dataloaders_with_partition(
        image_dir=image_dir,
        batch_size=8
)

model = SiameseNetwork().to(device)
criterion = ContrastiveLoss(margin=1.0)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

6000


In [4]:
# ==============================
# Evaluation: Cosine Similarity
# ==============================
def evaluate(model, loader, threshold=0.7):  # threshold closer to 1 for cosine similarity
    model.eval()
    y_true, y_pred, sims = [], [], []
    with torch.no_grad():
        for img1, img2, label in loader:
            img1, img2 = img1.to(device), img2.to(device)
            out1, out2 = model(img1, img2)
            sim = F.cosine_similarity(out1, out2).item()
            pred = 1 if sim > threshold else 0
            y_true.append(int(label.item()))
            y_pred.append(pred)
            sims.append(sim)
    acc = accuracy_score(y_true, y_pred)
    try:
        return acc, roc_auc_score(y_true, sims)
    except:
        return acc, None

# ==============================
# Training Loop (with PGD)
# ==============================
def train(model, train_loader, val_loader, criterion, optimizer, epochs=10, use_pgd=True, patience=3):
    best_val_acc = 0
    patience_counter = 0
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for img1, img2, label in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
            img1, img2, label = img1.to(device), img2.to(device), label.to(device)
            if use_pgd:
                img1_adv = pgd_attack(model, img1, img2, label, criterion, eps=0.03, alpha=0.01, iters=7)
                out1, out2 = model(img1_adv, img2)
            else:
                out1, out2 = model(img1, img2)
            loss = criterion(out1, out2, label)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        avg_loss = total_loss / len(train_loader)
        val_acc, _ = evaluate(model, val_loader, threshold=0.819)    
        print(f"Epoch [{epoch+1}/{epochs}] Avg Loss: {avg_loss:.4f} | Val Accuracy: {val_acc*100:.2f}%")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            patience_counter = 0
            torch.save(model.state_dict(), "models_bin/fr_pgd3.pth")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch+1}. Best Val Accuracy: {best_val_acc*100:.2f}%")
                break

In [ ]:
# ==============================
# Run Training and Evaluation
# ==============================
train(model, train_loader, val_loader, criterion, optimizer, epochs=10, use_pgd=False)

In [7]:
model = SiameseNetwork().to(device)
model_path = "models_bin/fr_pgd3.pth"
model.load_state_dict(torch.load(model_path, map_location=device))

<All keys matched successfully>

In [18]:
acc, roc_auc = evaluate(model, test_loader, threshold=0.87)
print("\nEvaluation (Cosine Similarity):")
print(f"Accuracy: {acc:.4f}")
print(f"ROC AUC: {roc_auc:.4f}" if roc_auc else "ROC AUC: N/A")


Evaluation (Cosine Similarity):
Accuracy: 0.7222
ROC AUC: 0.7787
